I'll poke at networkx and see what happens if I naively (since I have no clue what I'm doing) try to bring the RDF graph into networkx.

In [ ]:
from pathlib import Path
from rdflib import Graph

rdfpath = Path("../data/sidebar_graph.ttl")
g = Graph()
g.parse(rdfpath)
print(f"{len(g):,}")



352,428


In [11]:
from rdflib.extras.external_graph_libs import rdflib_to_networkx_graph

nxg = rdflib_to_networkx_graph(g)
print(f"Nodes: {len(nxg.nodes):,}")


Nodes: 198,459


In [34]:
node_list = [n for n in nxg.nodes]
from pprint import pprint
n = node_list[0]
print(f"type: {type(n)}")
print(f"__str__(): {n}")


type: <class 'rdflib.term.URIRef'>
__str__(): https://edh.ub.uni-heidelberg.de/edh/geographie/G031204


In [35]:
edge_list = [e for e in nxg.edges]
e = edge_list[0]
print(f"type: {type(n)}")
print(f"__str__(): {n}")

type: <class 'rdflib.term.URIRef'>
__str__(): https://edh.ub.uni-heidelberg.de/edh/geographie/G031204


In [37]:
neighbors = [neb for neb in nxg.neighbors(n)]
for neb in neighbors:
    print(neb)

https://www.geonames.org/2821164
Augusta Treverorum ({Landarmenhaus})
https://pleiades.stoa.org/places/108894
https://www.trismegistos.org/place/13685
https://purl.org/geojson/vocab#Feature


In [38]:
print(n)
print(len(neighbors))
for neb in neighbors:
    edge_d = nxg.get_edge_data(n, neb)
    pprint(edge_d)

https://edh.ub.uni-heidelberg.de/edh/geographie/G031204
5
{'triples': [(rdflib.term.URIRef('https://edh.ub.uni-heidelberg.de/edh/geographie/G031204'),
              rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#closeMatch'),
              rdflib.term.URIRef('https://www.geonames.org/2821164'))],
 'weight': 1}
{'triples': [(rdflib.term.URIRef('https://edh.ub.uni-heidelberg.de/edh/geographie/G031204'),
              rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#label'),
              rdflib.term.Literal('Augusta Treverorum ({Landarmenhaus})', lang='und'))],
 'weight': 1}
{'triples': [(rdflib.term.URIRef('https://edh.ub.uni-heidelberg.de/edh/geographie/G031204'),
              rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#closeMatch'),
              rdflib.term.URIRef('https://pleiades.stoa.org/places/108894'))],
 'weight': 1}
{'triples': [(rdflib.term.URIRef('https://edh.ub.uni-heidelberg.de/edh/geographie/G031204'),
              rdflib.term.URIRef('http:/

OK, so all the triples that connect two nodes are just banged into a single edge between them as an object. Since my primary interest now is trying to develop clusters of close matches that eventually point to Pleiades places, maybe I can process the RDF to get a single-mode, directional network to play with in networkx.

Will try doing SPARQL query to get just those triples that contain closeMatch 

In [39]:
query = """
SELECT DISTINCT ?s ?o
WHERE {
    ?s skos:closeMatch ?o
}
"""
results = g.query(query)
i = 0
for row in results:
    if i >= 10:
        break
    print(f"{row.s} closely matches {row.o}") # type: ignore
    i += 1


http://nomisma.org/id/agathopolis closely matches http://dbpedia.org/resource/Agathopolis
http://nomisma.org/id/agathopolis closely matches http://vocab.getty.edu/tgn/7017007
http://nomisma.org/id/agathopolis closely matches https://ikmk.smb.museum/ndp/ort/1948
http://nomisma.org/id/agathopolis closely matches https://pleiades.stoa.org/places/216721
http://www.wikidata.org/entity/Q341027 closely matches https://pleiades.stoa.org/places/216721
http://nomisma.org/id/alexandria_on_the_caucasus closely matches http://dbpedia.org/resource/Begram
http://nomisma.org/id/alexandria_on_the_caucasus closely matches https://pleiades.stoa.org/places/59668
https://vici.org/vici/10850 closely matches https://pleiades.stoa.org/places/59668
http://www.wikidata.org/entity/Q2005166 closely matches https://pleiades.stoa.org/places/59668
http://nomisma.org/id/alia closely matches http://collection.britishmuseum.org/id/place/x47036


In [ ]:
from rdflib.extras.external_graph_libs import rdflib_to_networkx_digraph # directed graph

